In [1]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [2]:
torch.manual_seed(42)
np.random.seed(42)


def load_videos_from_folder(folder, max_frames=50):   # we're extracting 50 frames
    videos = []
    labels = []
    
    for label in ["Violence", "NonViolence"]:
        path = os.path.join(folder, label)
        for filename in os.listdir(path):
            if filename.endswith(".mp4") or filename.endswith(".avi"):
                filepath = os.path.join(path, filename)
                cap = cv2.VideoCapture(filepath)
                frames = []
                
                while cap.isOpened():
                    ret, frame = cap.read()
                    if not ret:
                        break
                    frame = cv2.resize(frame, (64, 64))
                    frames.append(frame)
                    if len(frames) == max_frames:
                        break
                        
                cap.release()
                
                # Pad sequences to ensure they all have the same length
                while len(frames) < max_frames:
                    frames.append(np.zeros((64, 64, 3), dtype=np.uint8))
                videos.append(frames)
                labels.append(0 if label == "Violence" else 1)

    return np.array(videos), np.array(labels)

In [3]:
# Load the data
X, y = load_videos_from_folder("/kaggle/input/real-life-violence-situations-dataset/Real Life Violence Dataset", max_frames=20)

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Normalize pixel values to [0,1] and convert to float32
X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

In [4]:
'''PyTorch expects data in (batch, channels, height, width). Our videos are currently:
(num_videos, num_frames, height, width, channels). We'll convert this within our Dataset'''

class VideoDataset(Dataset):
    def __init__(self, videos, labels):
        self.videos = videos
        self.labels = labels
        
    def __len__(self):
        return len(self.videos)
    
    def __getitem__(self, idx):
        # Convert the video to a tensor and reorder axes: (T, H, W, C) -> (T, C, H, W)
        video = torch.tensor(self.videos[idx])
        video = video.permute(0, 3, 1, 2)
        label = torch.tensor(self.labels[idx]).float()
        return video, label

# Create dataset and dataloader objects
train_dataset = VideoDataset(X_train, y_train)
test_dataset = VideoDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [5]:
# PyTorch model
class VideoClassifier(nn.Module):
    def __init__(self):
        super(VideoClassifier, self).__init__()
        # Define a convolutional block to be applied to each frame
        self.conv_block = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 64 -> 32

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 32 -> 16

            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 16 -> 8

            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)   # 8 -> 4
        )
        # After conv layers: final feature map size = (16, 4, 4) → flattened to 16*4*4 = 256
        self.lstm = nn.LSTM(input_size=256, hidden_size=16, batch_first=True)
        self.fc1 = nn.Linear(16, 16)
        self.fc2 = nn.Linear(16, 1)


    def forward(self, x):
        # x shape: (batch, time, channels, height, width)
        batch_size, seq_len, C, H, W = x.size()
        # Merge batch and time dimensions for convolution
        x = x.view(batch_size * seq_len, C, H, W)
        x = self.conv_block(x)
        x = x.reshape(batch_size, seq_len, -1)  # shape: (batch, time, features)
        lstm_out, (h_n, _) = self.lstm(x)
        # Use the last hidden state as the representation
        out = h_n[-1]  # shape: (batch, hidden_size)
        out = self.fc1(out)
        out = torch.relu(out)
        out = self.fc2(out)
        out = torch.sigmoid(out)
        return out.squeeze()


# Instantiate model, loss, and optimizer
model = VideoClassifier()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [6]:
# Training loop
num_epochs = 15
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for videos, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * videos.size(0)
    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")
    

# Evaluation on test data
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for videos, labels in test_loader:
        outputs = model(videos)
        preds = (outputs > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Epoch 1/15, Loss: 0.6933
Epoch 2/15, Loss: 0.6848
Epoch 3/15, Loss: 0.6716
Epoch 4/15, Loss: 0.6795
Epoch 5/15, Loss: 0.6470
Epoch 6/15, Loss: 0.6380
Epoch 7/15, Loss: 0.6032
Epoch 8/15, Loss: 0.5616
Epoch 9/15, Loss: 0.5168
Epoch 10/15, Loss: 0.4815
Epoch 11/15, Loss: 0.4364
Epoch 12/15, Loss: 0.3969
Epoch 13/15, Loss: 0.3763
Epoch 14/15, Loss: 0.3245
Epoch 15/15, Loss: 0.2990
Test Accuracy: 80.00%


In [7]:
# Save the trained model
torch.save(model, "violence.pt")
print("Model saved as violence.pt")

Model saved as violence.pt
